In [1]:
!pip install xlrd
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 116.8 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import nltk
nltk.download('stopwords')

from google.colab import drive
drive.mount('/content/gdrive')
directorio = "/content/gdrive/MyDrive/nlp/"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Mounted at /content/gdrive


In [3]:
import networkx as nx
import os
import xml.etree.ElementTree as ET
import nltk
import string
import spacy
from operator import itemgetter
nlp = spacy.load("es_core_news_sm")

In [4]:
import xlrd
def Construye_Grafo():
  nombre_archivo= "NAP.xls"
  libro = xlrd.open_workbook(directorio + nombre_archivo)
  hoja = libro.sheet_by_index(0)
  GrafoFrec = nx.Graph()
  GrafoTiempo = nx.Graph()
  GrafoAsoc = nx.Graph()
  num_filas = hoja.nrows
  estimulo = ""
  lema = ""
  frecuencia = 0
  tiempo = 0
  asociado = 0
  for fila in range(0, num_filas-1):
    celda = hoja.cell(fila,0)
    cadena = celda.value
    if cadena == "--PALABRAS--":
      continue
    elif cadena == "======":
      estimulo = hoja.cell(fila+1,4).value
    elif cadena == "" or cadena == "=":
      continue
    elif hoja.cell(fila,2).value == "":
      continue
    else:
      frecuencia = hoja.cell(fila,1).value
      tiempo = hoja.cell(fila,2).value
      asociado = hoja.cell(fila,3).value
      palabra = hoja.cell(fila,4).value
      estimulo = estimulo.strip()
      lema = str(palabra)
      lema = lema.strip()
      GrafoFrec.add_edge(estimulo, lema, weight=1/float(frecuencia), sim=float(frecuencia))
      GrafoTiempo.add_edge(estimulo, lema, weight=float(tiempo), sim = 1/(float(tiempo)+1))
      GrafoAsoc.add_edge(estimulo, lema, weight=100-float(asociado), sim=float(asociado))
  return GrafoFrec, GrafoTiempo, GrafoAsoc

In [5]:
def limpia_lematiza(cadena):
  limpiado = ""
  palabras_funcionales = nltk.corpus.stopwords.words('spanish')
  for c in string.punctuation:
    cadena = cadena.replace(c, "")
  cadena = cadena.strip()
  for palabra in cadena.split():
    if palabra not in palabras_funcionales:
      doc = nlp(palabra)
      limpiado += doc[0].lemma_ + " "
  return limpiado

In [6]:
def conceptos(lista,subconjunto):
  datos = []
  if len(lista) <= 100:
    tope = len(lista)
  else:
    tope = 100
  for x in range(0,tope):
    if str(lista[x][0]) not in subconjunto and float(lista[x][1])>0:
      datos.append(str(lista[x][0]))
  return datos

In [7]:
def precision(k,arreglo,concepto):
  tope = k
  encontrado = 0
  if k> len(arreglo):
    tope = len(arreglo)
  for i in range (0,tope):
    if arreglo[i] == concepto.strip():
      encontrado = 1
  return encontrado

In [8]:
def reducir_grafo(subconjunto,grafo_original):
  grafo_reducido = nx.Graph()
  nuevos_nodos = []
  nuevos_nodos.extend(subconjunto)
  for nodo in subconjunto:
    vecinos = grafo_original[nodo]
    for vecino in vecinos:
      nuevos_nodos.append(vecino)
  grafo_reducido = grafo_original.subgraph(nuevos_nodos)
  return grafo_reducido

In [9]:
def diccionario_cc(grafo,definicion):
  texto = limpia_lematiza(definicion)
  texto = texto.lower()
  tokens = texto.split(" ")
  subconjunto_lemas = []
  for token in tokens:
    if token in grafo.nodes() and token!= "":
      subconjunto_lemas.append(token)
  if len(subconjunto_lemas) > 0:
    grafo_reducido = reducir_grafo(subconjunto_lemas,grafo)
    resultado = nx.closeness_centrality(grafo_reducido,distance="weight")
    encontrado = sorted(resultado.items(), key=lambda x: x[1], reverse=True)[0:99]
    return conceptos(encontrado,subconjunto_lemas)
  else:
    return []

In [10]:
def diccionario_btc(grafo,definicion):
  texto = limpia_lematiza(definicion)
  texto = texto.lower()
  tokens = texto.split(" ")
  subconjunto_lemas = []
  for token in tokens:
    if token in grafo.nodes() and token!= "":
      subconjunto_lemas.append(token)
  if len(subconjunto_lemas) > 0:
    resultado_asociacion = nx.betweenness_centrality_subset(grafo,subconjunto_lemas,subconjunto_lemas,weight="weight",normalized=True)
    encontrado = sorted(resultado_asociacion.items(), key=lambda x: x[1], reverse=True)[0:99]
    return conceptos(encontrado,subconjunto_lemas)
  else:
    return []

In [11]:
def diccionario_dc(grafo,definicion):
  texto = limpia_lematiza(definicion)
  texto = texto.lower()
  tokens = texto.split(" ")
  subconjunto_lemas = []
  for token in tokens:
    if token in grafo.nodes() and token!= "":
      subconjunto_lemas.append(token)
  if len(subconjunto_lemas) > 0:
    grafo_reducido = reducir_grafo(subconjunto_lemas,grafo)
    resultado = nx.degree_centrality(grafo_reducido)
    encontrado = sorted(resultado.items(), key=lambda x: x[1], reverse=True)[0:99]
    return conceptos(encontrado,subconjunto_lemas)
  else:
    return []

In [19]:
def diccionario_pr(grafo, definicion, alpha=0.5):
  texto = limpia_lematiza(definicion)
  texto = texto.lower()
  tokens = texto.split(" ")
  subconjunto_lemas = []
  for token in tokens:
    if token in grafo.nodes() and token != "":
      subconjunto_lemas.append(token)
  if len(subconjunto_lemas) > 0:
    pers = {n: (1 if n in subconjunto_lemas else 0) for n in grafo.nodes()}
    resultado = nx.pagerank(grafo, alpha=alpha, personalization=pers, weight="sim")
    encontrado = sorted(resultado.items(), key=lambda x: x[1], reverse=True)[0:99]
    return conceptos(encontrado, subconjunto_lemas)
  else:
    return []

In [22]:
def diccionario_btc_reducido(grafo,definicion):
  texto = limpia_lematiza(definicion)
  texto = texto.lower()
  tokens = texto.split(" ")
  subconjunto_lemas = []
  for token in tokens:
    if token in grafo.nodes() and token!= "":
      subconjunto_lemas.append(token)
  if len(subconjunto_lemas) > 0:
    grafo_reducido = reducir_grafo(subconjunto_lemas,grafo)
    resultado = nx.betweenness_centrality(grafo_reducido,weight="weight",normalized=True)
    encontrado = sorted(resultado.items(), key=lambda x: x[1], reverse=True)[0:99]
    return conceptos(encontrado,subconjunto_lemas)
  else:
    return []

In [23]:
def rendimiento_mdbl_btc_reducido():
  grafo_frec, grafo_tiempo, grafo_asociacion = Construye_Grafo()
  p_1_t = p_1_f = p_1_a = 0
  p_3_t = p_3_f = p_3_a = 0
  p_5_t = p_5_f = p_5_a = 0
  total = 0
  for archivo in os.listdir(directorio+"corpus/"):
    #print (archivo)
    datos = open(directorio+"corpus/"+archivo,encoding="utf8")
    lineas = datos.readlines()
    concepto = str(lineas[0]).lower()
    for linea in lineas[1:]:
      total += 1
      candidatos = diccionario_btc_reducido(grafo_asociacion,linea)
      p_1_a += precision(1,candidatos,concepto)
      p_3_a += precision(3,candidatos,concepto)
      p_5_a += precision(5,candidatos,concepto)
      candidatos = diccionario_btc_reducido(grafo_frec,linea)
      p_1_f += precision(1,candidatos,concepto)
      p_3_f += precision(3,candidatos,concepto)
      p_5_f += precision(5,candidatos,concepto)
      candidatos = diccionario_btc_reducido(grafo_tiempo,linea)
      p_1_t += precision(1,candidatos,concepto)
      p_3_t += precision(3,candidatos,concepto)
      p_5_t += precision(5,candidatos,concepto)
  print("Asociación")
  print("p@1:",p_1_a/total,"p@3:",p_3_a/total,"p@5:",p_5_a/total)
  print("Frecuencia")
  print("p@1:",p_1_f/total,"p@3:",p_3_f/total,"p@5:",p_5_f/total)
  print("Tiempo")
  print("p@1:",p_1_t/total,"p@3:",p_3_t/total,"p@5:",p_5_t/total)
  return

In [14]:
GrafoFrec, GrafoTiempo, GrafoAsoc = Construye_Grafo()
print(GrafoAsoc.number_of_nodes(), GrafoAsoc.number_of_edges())
print(list(GrafoAsoc["abeja"].items())[:3])

4239 12872
[('miel', {'weight': 50.18182, 'sim': 49.81818}), ('picar', {'weight': 91.636364, 'sim': 8.363636}), ('picadura', {'weight': 93.454545, 'sim': 6.545455})]


rendimiento générique rapide à travailler

In [15]:
def rendimiento(modelo, grafo):
  p1 = p3 = p5 = total = 0
  for archivo in os.listdir(directorio+"corpus/"):
    lineas = open(directorio+"corpus/"+archivo, encoding="utf8").readlines()
    concepto = str(lineas[0]).lower()
    for linea in lineas[1:]:
      total += 1
      candidatos = modelo(grafo, linea)
      p1 += precision(1, candidatos, concepto)
      p3 += precision(3, candidatos, concepto)
      p5 += precision(5, candidatos, concepto)
  print("p@1:", p1/total, "p@3:", p3/total, "p@5:", p5/total)

In [20]:
rendimiento(diccionario_dc, GrafoAsoc)    # degré
rendimiento(diccionario_cc, GrafoAsoc)    # proximité
rendimiento(diccionario_btc, GrafoAsoc)   # intermédiarité (la plus lente, quelques minutes)

p@1: 0.11594202898550725 p@3: 0.25 p@5: 0.32971014492753625
p@1: 0.2210144927536232 p@3: 0.427536231884058 p@5: 0.5181159420289855
p@1: 0.2644927536231884 p@3: 0.4057971014492754 p@5: 0.44565217391304346


In [24]:
rendimiento_mdbl_btc_reducido()

Asociación
p@1: 0.16666666666666666 p@3: 0.36231884057971014 p@5: 0.4492753623188406
Frecuencia
p@1: 0.13768115942028986 p@3: 0.3007246376811594 p@5: 0.36231884057971014
Tiempo
p@1: 0.08333333333333333 p@3: 0.20652173913043478 p@5: 0.2608695652173913


In [21]:
rendimiento(diccionario_pr, GrafoFrec)
rendimiento(diccionario_pr, GrafoTiempo)
rendimiento(diccionario_cc, GrafoFrec)
rendimiento(diccionario_cc, GrafoTiempo)

p@1: 0.42391304347826086 p@3: 0.605072463768116 p@5: 0.6956521739130435
p@1: 0.1956521739130435 p@3: 0.3804347826086957 p@5: 0.5289855072463768
p@1: 0.1956521739130435 p@3: 0.35507246376811596 p@5: 0.5144927536231884
p@1: 0.07608695652173914 p@3: 0.213768115942029 p@5: 0.286231884057971


In [25]:
def errores(modelo, grafo, max_errores=10):
  n = 0
  for archivo in os.listdir(directorio+"corpus/"):
    lineas = open(directorio+"corpus/"+archivo, encoding="utf8").readlines()
    concepto = str(lineas[0]).strip().lower()
    for linea in lineas[1:]:
      cand = modelo(grafo, linea)
      if concepto not in cand[:5]:
        print("Concepto:", concepto)
        print("Descripción:", linea.strip())
        print("Top 5:", cand[:5], "\n")
        n += 1
        if n >= max_errores:
          return

errores(diccionario_pr, GrafoAsoc)

Concepto: dulce
Descripción: Esa sensacion que produce comer azucar.
Top 5: ['globo', 'tenedor', 'comida', 'cuchara', 'aire'] 

Concepto: gelatina
Descripción: La que se hace con agua y cuaja
Top 5: ['plastilina', 'burbuja', 'palo', 'manguera', 'cubeta'] 

Concepto: cochino
Descripción: con el que se hacen las carnitas
Top 5: ['plastilina', 'palo', 'burbuja', 'escoba', 'moldear'] 

Concepto: aspiradora
Descripción: electrodoméstico que sirve para juntar polvo
Top 5: ['familia', 'aspirador', 'plato', 'lavadora', 'plancha'] 

Concepto: aspiradora
Descripción: Escoba eléctrica
Top 5: ['barrer', 'tren', 'cepillo_de_dientes', 'timbre', 'palo'] 

Concepto: aspiradora
Descripción: Aparato para sacar el polvo
Top 5: ['aspirador', 'pala', 'calabaza', 'cámara', 'televisión'] 

Concepto: aspiradora
Descripción: aparato electrónico para quitar el polvo
Top 5: ['aspirador', 'plancha', 'calzón', 'bota', 'lavadora'] 

Concepto: aspiradora
Descripción: es un aparato para limpiar y succionar el polvo
T